In [ ]:
import os
import warnings
from itertools import cycle
from typing import Dict, List, Optional, Sequence, Tuple

import celltypist
import matplotlib as mpl
import matplotlib.colors as mc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from celltypist import models, train
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path
from scipy.stats import mannwhitneyu, ttest_ind

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42

warnings.filterwarnings("ignore")

os.makedirs("figures", exist_ok=True)
os.makedirs("models", exist_ok=True)


# Fig4a

In [ ]:
MAIGD_T = sc.read_h5ad( 'MAIGD_T.h5ad')


In [ ]:
base_cmap = mpl.colormaps["coolwarm"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_coolwarm",
    base_cmap(np.linspace(0.10, 0.90, 256))
)


In [ ]:
UCT_color_map = {
    "MAI_T_c1": "#c5b0d5",
    "GD_TEM_c2":  "#ffff99",
    "GD_TEMRA_c3":  "#ff8a65",
}


In [ ]:
ax = sc.pl.umap(
    MAIGD_T,
    color="anno_UCT",
    palette=UCT_color_map,
    use_raw=False,
    frameon=False,
    size=25,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('figures/Fig4a_UCT_umap.pdf', dpi=600, bbox_inches="tight")


# Fig4b

In [ ]:
marker_genes_dict = {
    "CD8/CD4":["CD8A","CD8B","CD4"],
    "MAIT":["TRAV1-2","SLC4A10"],
    "GDT":["TRDV2","TRGV9"],
    "innate-like":["ZBTB16","KLRB1"],
    "Cytolytic": ["GZMK","GZMB","PRF1","GNLY"],
    "Terminal":["ZEB2","TBX21"],
    "Type3":["RORC","CCR6"],
    }


In [ ]:
available_genes = set(MAIGD_T.var_names)

filtered_marker_genes_dict = {
    category: [gene for gene in genes if gene in available_genes]
    for category, genes in marker_genes_dict.items()
}

filtered_marker_genes_dict = {
    k: v for k, v in filtered_marker_genes_dict.items() if v
}

if not filtered_marker_genes_dict:
    raise ValueError("没有找到可以用于绘图的基因，请检查 marker_genes_dict 和数据集中的基因名是否匹配。")

ax=sc.pl.dotplot(
    MAIGD_T,
    filtered_marker_genes_dict,
    'anno_UCT',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.savefig('figures/Fig4b_UCT_markers.pdf', bbox_inches="tight")
ax.show()


# Fig4c

In [ ]:
def compute_celltype_fraction_per_sample(
    obs_df: pd.DataFrame,
    *,
    sample_id_col: str = "Sample_ID",
    meta_cols=None,
    celltype_col: str = "anno_TKA"
):

    if meta_cols is None:
        meta_cols = ["Disease"]

    sample_meta = (
        obs_df[[sample_id_col] + meta_cols]
        .dropna(subset=[sample_id_col])
        .drop_duplicates(subset=[sample_id_col])
    )

    counts = (
        obs_df
        .groupby([sample_id_col, celltype_col])
        .size()
        .reset_index(name="n")
    )

    totals = (
        counts
        .groupby(sample_id_col)["n"]
        .sum()
        .reset_index(name="total")
    )

    frac_df = counts.merge(
        totals,
        on=sample_id_col,
        how="left"
    )

    frac_df["frac"] = frac_df["n"] / frac_df["total"]

    frac_df = frac_df.merge(
        sample_meta,
        on=sample_id_col,
        how="left"
    )

    frac_df = frac_df[
        [sample_id_col]
        + meta_cols
        + [celltype_col, "n", "total", "frac"]
    ]

    count_matrix = (
        counts
        .pivot(
            index=sample_id_col,
            columns=celltype_col,
            values="n"
        )
        .fillna(0)
        .astype(int)
    )

    count_matrix.columns.name = None

    count_matrix["total"] = (
        totals
        .set_index(sample_id_col)["total"]
    )

    count_matrix = count_matrix.join(
        sample_meta.set_index(sample_id_col),
        how="left"
    )

    return frac_df, count_matrix


In [ ]:
def plot_group_box_dots(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    *,
    group_order: Optional[Sequence[str]] = None,
    palette: Optional[Dict[str, str]] = None,
    subgroup_col: Optional[str] = None,
    subgroup_order: Optional[Sequence[str]] = None,
    subgroup_palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,
    figsize: Tuple[float, float] = (6, 4),
    jitter: float = 0.08,
    point_size: float = 20,
    min_box_height_ratio: float = 0.03,
    random_state: int = 0,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    title: Optional[str] = None,
    show_subgroup_legend: bool = True,
    box_edge_color: str = "#C0C0C0",
    box_line_alpha: float = 0.6,
    box_line_width: float = 1.2,
):
    rng = np.random.default_rng(random_state)

    if group_order is None:
        group_order = list(pd.unique(df[group_col].dropna()))
    else:
        group_order = [g for g in group_order if g in df[group_col].values]

    use_cols = [group_col, value_col]
    if subgroup_col is not None:
        use_cols.append(subgroup_col)

    sub = df[use_cols].dropna(subset=[group_col, value_col])
    if sub.empty:
        raise ValueError("筛选后数据为空，请检查 value_col / group_col / subgroup_col。")

    all_vals = sub[value_col].values.astype(float)
    y_min, y_max = float(np.min(all_vals)), float(np.max(all_vals))
    if y_min == y_max:
        y_min -= 0.5
        y_max += 0.5
    y_range = y_max - y_min
    min_box_h = y_range * min_box_height_ratio

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    if palette is None:
        palette = {}
        if meta_color_map is not None:
            for g in group_order:
                if g in meta_color_map:
                    palette[g] = meta_color_map[g]
        for g in group_order:
            if g not in palette:
                palette[g] = box_edge_color

    if subgroup_col is not None:
        if subgroup_order is None:
            sub_levels = list(pd.unique(sub[subgroup_col].dropna()))
        else:
            sub_levels = [s for s in subgroup_order if s in sub[subgroup_col].values]

        final_subgroup_palette = {}

        if meta_color_map is not None:
            for s in sub_levels:
                if s in meta_color_map:
                    final_subgroup_palette[s] = meta_color_map[s]

        if subgroup_palette is not None:
            for s in sub_levels:
                if s in subgroup_palette:
                    final_subgroup_palette[s] = subgroup_palette[s]

        missing_levels = [s for s in sub_levels if s not in final_subgroup_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(
                missing_levels,
                used_colors=final_subgroup_palette.values()
            )
            final_subgroup_palette.update(auto_pal)

        subgroup_palette = final_subgroup_palette
    else:
        subgroup_palette = {}
        sub_levels = []

    fig, ax = plt.subplots(figsize=figsize)
    xs = np.arange(len(group_order)) * 0.5
    box_width = 0.30

    whisker_cap_width = box_width / 4
    median_width = box_line_width
    whisker_width = box_line_width
    cap_width = box_line_width
    rect_line_width = box_line_width

    for x, g in zip(xs, group_order):
        g_mask = sub[group_col] == g
        vals = sub.loc[g_mask, value_col].values.astype(float)
        if len(vals) == 0:
            continue

        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        if len(vals) >= 2:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
        else:
            med = vals[0]
            q1 = med - min_box_h / 2
            q3 = med + min_box_h / 2

        if (q3 - q1) < min_box_h:
            center = (q1 + q3) / 2
            q1 = center - min_box_h / 2
            q3 = center + min_box_h / 2

        w_low = float(np.min(vals))
        w_high = float(np.max(vals))

        box_col = palette.get(g, box_edge_color)

        rect = plt.Rectangle(
            (x - box_width / 2, q1),
            box_width,
            q3 - q1,
            facecolor="none",
            edgecolor=box_col,
            linewidth=rect_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.add_patch(rect)

        ax.plot(
            [x - box_width / 2, x + box_width / 2],
            [med, med],
            color=box_col,
            linewidth=median_width,
            alpha=box_line_alpha,
            zorder=2,
        )

        ax.plot(
            [x, x], [q3, w_high],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x, x], [q1, w_low],
            color=box_col,
            linewidth=whisker_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_high, w_high],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_low, w_low],
            color=box_col,
            linewidth=cap_width,
            alpha=box_line_alpha,
            zorder=1
        )

        if subgroup_col is None:
            jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(vals))
            ax.scatter(
                np.full_like(vals, x, dtype=float) + jitter_x,
                vals,
                s=point_size,
                color=box_col,
                alpha=0.85,
                edgecolors="none",
                zorder=3,
            )
        else:
            g_df = sub.loc[g_mask, [value_col, subgroup_col]]
            for sub_name, sdf in g_df.groupby(subgroup_col):
                v = sdf[value_col].values.astype(float)
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue

                jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(v))
                col = subgroup_palette.get(sub_name, "#555555")

                ax.scatter(
                    np.full_like(v, x, dtype=float) + jitter_x,
                    v,
                    s=point_size,
                    color=col,
                    alpha=0.9,
                    edgecolors="none",
                    zorder=3,
                    label=sub_name,
                )

    ax.set_xticks(xs)
    ax.set_xticklabels(group_order, rotation=45, ha="right")
    ax.set_ylim(y_min, y_max)
    ax.margins(x=0.05)

    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if title is not None:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if subgroup_col is not None and show_subgroup_legend:
        handles, labels = ax.get_legend_handles_labels()
        handle_by_label = dict(zip(labels, handles))

        if subgroup_order is not None:
            legend_labels = [s for s in subgroup_order if s in handle_by_label]
        else:
            legend_labels = [s for s in sub_levels if s in handle_by_label]

        legend_handles = [handle_by_label[l] for l in legend_labels]

        ax.legend(
            legend_handles,
            legend_labels,
            title=subgroup_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
            fontsize="x-small"
        )

    plt.tight_layout()
    return fig, ax


In [ ]:
T_atlas = sc.read_h5ad( 'T.h5ad')


In [ ]:
T_atlas_tumor= T_atlas[T_atlas.obs['Sample_Type'].isin(['T'])].copy()


In [ ]:
frac_df,count_matrix  = compute_celltype_fraction_per_sample(
    T_atlas_tumor.obs,
    sample_id_col="Sample_ID",
    celltype_col="T_sub_level_1",
    meta_cols=["Disease","Sample_Type","Period_merged"],
)


In [ ]:
meta_color_map = {
    "Child": "#fdae6b",
    "Adult": "#9ecae1",
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}

disease_order = ['pLGG','pHGG','MB','NB','WT','HB','PB',
                'aHCC','BRCA','CRC','HCC','LC','RC','PC']


In [ ]:
frac_df_sub = frac_df.loc[frac_df["T_sub_level_1"] == "MAI_T"].copy()
fig, ax = plot_group_box_dots(
    df=frac_df_sub,
    value_col="frac",
    group_col="Period_merged",
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(3.5, 4),
    ylabel='Proportion',
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/Fig4c_MAIT_proportion.pdf', dpi=300, bbox_inches="tight")
plt.show()

group_col = "Period_merged"
value_col = "frac"

df = frac_df_sub[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

def q(x, q): return float(np.quantile(x, q)) if len(x) else np.nan
summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


In [ ]:
frac_df_sub = frac_df.loc[frac_df["T_sub_level_1"] == "GD_T"].copy()
fig, ax = plot_group_box_dots(
    df=frac_df_sub,
    value_col="frac",
    group_col="Period_merged",
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(3.5, 4),
    ylabel='Proportion',
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/Fig4c_GDT_proportion.pdf', dpi=300, bbox_inches="tight")
plt.show()

group_col = "Period_merged"
value_col = "frac"

df = frac_df_sub[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


In [ ]:
frac_df_sub = frac_df.loc[frac_df["T_sub_level_1"] == "IM_T"].copy()
fig, ax = plot_group_box_dots(
    df=frac_df_sub,
    value_col="frac",
    group_col="Period_merged",
    group_order=["Adult","Child"],
    subgroup_col="Disease",
    subgroup_order=disease_order,
    figsize=(3.5, 4),
    ylabel='Proportion',
    meta_color_map=meta_color_map,
    box_line_alpha=0.3,
    box_line_width=1.2,

)
plt.savefig('figures/Fig4c_IMT_proportion.pdf', dpi=300, bbox_inches="tight")
plt.show()

group_col = "Period_merged"
value_col = "frac"

df = frac_df_sub[[group_col, value_col]].dropna().copy()

groups = df[group_col].dropna().drop_duplicates().tolist()
if len(groups) != 2:
    raise ValueError(f"{group_col} 应该正好有两类，当前为 {len(groups)}：{groups}")

g1, g2 = groups
A = df.loc[df[group_col] == g1, value_col].to_numpy(float)
B = df.loc[df[group_col] == g2, value_col].to_numpy(float)

if A.size == 0 or B.size == 0:
    raise ValueError("两组中至少有一组没有有效数据。")

U, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"[Mann–Whitney U] {g1} vs {g2}: U={U:.4g}, p={p:.4g}, n1={A.size}, n2={B.size}")

summary = pd.DataFrame({
    "group": [g1, g2],
    "n": [A.size, B.size],
    "median": [np.median(A), np.median(B)],
    "Q1": [q(A, 0.25), q(B, 0.25)],
    "Q3": [q(A, 0.75), q(B, 0.75)],
})
summary["IQR"] = summary["Q3"] - summary["Q1"]
print("\nSummary:")
print(summary.to_string(index=False))


# Fig4d

In [ ]:
def plot_box_by_disease(
    frac_df: pd.DataFrame,
    *,
    disease_col: str = "Disease",
    sample_id_col: str = "Sample_ID",
    celltype_col: str = "annotation_TT",
    celltypes: List[str] = None,
    disease_order: List[str] = None,
    figsize: Tuple[float, float] = None,
    show_pct: bool = True,
    random_state: int = 0,
    min_box_height_ratio: float = 0.03,

    palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,

    all_celltypes_for_palette: Optional[List[str]] = None,
    obs_celltype_series: Optional[pd.Series] = None,
    pair_tests: Optional[List[Tuple[str, str]]] = None,
    test_method: str = "mannwhitney",

    sig_line_y_ratio: float = 0.05,
    sig_text_offset_ratio: float = 0.01,
    sig_line_pad: float = 0.05,
):
    rng = np.random.default_rng(random_state)

    if celltypes is None:
        celltypes = frac_df[celltype_col].unique().tolist()

    if disease_order is None:
        default_order = [
            "MB", "pLGG", "pHGG", 'aHCC', "WT",
            'RC', "HB", "HCC", 'PB', 'PC',
            "NB", "BRCA", "CRC", "LC"
        ]
        present = frac_df[disease_col].dropna().unique().tolist()
        disease_order = [d for d in default_order if d in present]

    n_ct = len(celltypes)
    if figsize is None:
        figsize = (max(8, len(disease_order) * 0.7), 2.3 * n_ct)

    if palette is None:
        if all_celltypes_for_palette is not None:
            all_ct = [ct for ct in all_celltypes_for_palette
                      if ct in frac_df[celltype_col].unique()]
        elif obs_celltype_series is not None:
            s = obs_celltype_series
            if hasattr(s, "cat"):
                try:
                    cats = list(s.cat.categories)
                    all_ct = [ct for ct in cats if ct in frac_df[celltype_col].unique()]
                except Exception:
                    all_ct = list(pd.unique(s))
            else:
                all_ct = list(pd.unique(s))
                all_ct = [ct for ct in all_ct if ct in frac_df[celltype_col].unique()]
        else:
            all_ct = frac_df[celltype_col].unique().tolist()

        palette = {}

        if meta_color_map is not None:
            for ct in all_ct:
                if ct in meta_color_map:
                    palette[ct] = meta_color_map[ct]

        missing_ct = [ct for ct in all_ct if ct not in palette]
        if len(missing_ct) > 0:
            raw_colors = (
                list(cm.Greys(np.linspace(0.2, 0.4, 2))) +
                list(cm.Greens(np.linspace(0.2, 0.6, 7))) +
                list(cm.Oranges(np.linspace(0.2, 0.4, 7))) +
                list(cm.Reds(np.linspace(0.4, 0.6, 4))) +
                list(cm.Purples(np.linspace(0.4, 0.6, 1))) +
                list(cm.Blues(np.linspace(0.4, 0.4, 1))) +
                list(cm.YlOrBr(np.linspace(0.2, 0.6, 1)))
            )

            def to_morandi(c, mix=0.8, grey=(0.88, 0.86, 0.84)):
                r, g, b, a = c
                return (
                    r * mix + grey[0] * (1 - mix),
                    g * mix + grey[1] * (1 - mix),
                    b * mix + grey[2] * (1 - mix),
                    a
                )

            color_pool = [to_hex(to_morandi(c)) for c in raw_colors]
            used_colors = set(c.lower() for c in palette.values())
            color_pool = [c for c in color_pool if c.lower() not in used_colors]

            if len(color_pool) == 0:
                color_pool = [to_hex(to_morandi(c)) for c in raw_colors]

            if len(missing_ct) > len(color_pool):
                fill_colors = [next(cycle(color_pool)) for _ in missing_ct]
            else:
                fill_colors = color_pool[:len(missing_ct)]

            for ct, c in zip(missing_ct, fill_colors):
                palette[ct] = c

    ct_color = {ct: palette.get(ct, "#808080") for ct in celltypes}

    if pair_tests is None:
        pair_tests = [
            ("pHGG", 'aHCC'),
            ("WT", 'RC'),
            ("HB", "HCC"),
            ('PB', 'PC'),
        ]

    fig, axes = plt.subplots(n_ct, 1, figsize=figsize, sharex=False)
    if n_ct == 1:
        axes = [axes]

    xs = np.arange(len(disease_order)) * 0.6

    for ax, ct in zip(axes, celltypes):
        sub = frac_df.loc[frac_df[celltype_col] == ct].copy()
        if sub.empty:
            ax.text(0.5, 0.5, f"No data for {ct}", ha="center", va="center")
            ax.axis("off")
            continue

        if show_pct:
            sub["value"] = sub["frac"] * 100
        else:
            sub["value"] = sub["frac"]

        all_vals = sub["value"].values
        if len(all_vals) == 0:
            y_min, y_max = 0.0, 1.0
        else:
            y_min, y_max = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
            if y_min == y_max:
                y_min -= 0.5
                y_max += 0.5

        y_range = y_max - y_min
        min_box_h = y_range * min_box_height_ratio if y_range > 0 else 0.1

        box_data = [
            sub.loc[sub[disease_col] == d, "value"].values
            for d in disease_order
        ]

        col = ct_color.get(ct, "#808080")
        box_width = 0.5

        for x, vals in zip(xs, box_data):
            if len(vals) == 0:
                continue

            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue

            if len(vals) >= 2:
                q1, med, q3 = np.percentile(vals, [25, 50, 75])
            else:
                med = vals[0]
                q1 = med - min_box_h / 2
                q3 = med + min_box_h / 2

            if (q3 - q1) < min_box_h:
                center = (q1 + q3) / 2
                q1 = center - min_box_h / 2
                q3 = center + min_box_h / 2

            w_low = float(np.min(vals))
            w_high = float(np.max(vals))

            rect = plt.Rectangle(
                (x - box_width / 2, q1),
                box_width,
                q3 - q1,
                facecolor="none",
                edgecolor=col,
                linewidth=1.0,
                zorder=1
            )
            ax.add_patch(rect)

            ax.plot(
                [x - box_width / 2, x + box_width / 2],
                [med, med],
                color=col, linewidth=1.2, zorder=2
            )

            ax.plot([x, x], [q3, w_high], color=col, linewidth=1.0, zorder=1)
            ax.plot([x, x], [q1, w_low], color=col, linewidth=1.0, zorder=1)
            ax.plot(
                [x - box_width / 4, x + box_width / 4],
                [w_high, w_high],
                color=col, linewidth=1.0, zorder=1
            )
            ax.plot(
                [x - box_width / 4, x + box_width / 4],
                [w_low, w_low],
                color=col, linewidth=1.0, zorder=1
            )

        for x, d in zip(xs, disease_order):
            vals = sub.loc[sub[disease_col] == d, "value"].values
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue
            jitter = rng.normal(loc=0.0, scale=0.08, size=len(vals))
            ax.scatter(
                x + jitter,
                vals,
                s=14,
                alpha=0.85,
                color=col,
                edgecolors="none",
                zorder=3
            )

        max_annot_y = y_max
        if test_method.lower() != "none" and y_range > 0:
            line_y = y_max + sig_line_y_ratio * y_range
            text_y = line_y + sig_text_offset_ratio * y_range

            for (d1, d2) in pair_tests:
                if (d1 not in disease_order) or (d2 not in disease_order):
                    continue

                v1 = sub.loc[sub[disease_col] == d1, "value"].values
                v2 = sub.loc[sub[disease_col] == d2, "value"].values
                v1 = v1[np.isfinite(v1)]
                v2 = v2[np.isfinite(v2)]

                if len(v1) < 2 or len(v2) < 2:
                    continue

                if test_method.lower() == "mannwhitney":
                    p = mannwhitneyu(v1, v2, alternative="two-sided").pvalue
                elif test_method.lower() == "ttest":
                    p = ttest_ind(v1, v2, equal_var=False).pvalue
                else:
                    continue

                if p <= 0.0001:
                    label = "****"
                elif p <= 0.001:
                    label = "***"
                elif p <= 0.01:
                    label = "**"
                elif p <= 0.05:
                    label = "*"
                else:
                    label = "ns"

                i1 = disease_order.index(d1)
                i2 = disease_order.index(d2)
                x1, x2 = xs[i1], xs[i2]
                x_left, x_right = min(x1, x2), max(x1, x2)

                ax.plot(
                    [x_left + sig_line_pad, x_right - sig_line_pad],
                    [line_y, line_y],
                    color="black",
                    linewidth=0.8,
                    zorder=4
                )

                ax.text(
                    (x_left + x_right) / 2.0,
                    text_y,
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    zorder=5
                )

                max_annot_y = max(max_annot_y, text_y + 0.03 * y_range)

        ax.set_ylim(y_min, max_annot_y)

        ax.set_ylabel("Proportion (%)")
        ax.set_title(ct, fontsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for ax in axes:
        ax.set_xticks(xs)
        ax.set_xticklabels(disease_order, rotation=45, ha="right")

    plt.tight_layout(h_pad=1.0)
    return fig, axes


In [ ]:
celltypes_to_plot = ["GD_T","IM_T","MAI_T"]
disease_order = ["NB","MB",'pLGG','pHGG','aHCC',"WT",
            'RC',"HB","HCC",'PB','PC'
            ,"BRCA","CRC","LC"]
pairs = [
    ('pHGG', 'aHCC'),
    ("WT",   'RC'),
    ("HB",   "HCC"),
    ('PB',  'PC'),
]

fig, axes = plot_box_by_disease(
    frac_df,
    disease_col="Disease",
    sample_id_col="Sample_ID",
    celltype_col="T_sub_level_1",
    celltypes=celltypes_to_plot,
    disease_order=disease_order,
    pair_tests=pairs,
    obs_celltype_series=T_atlas.obs["T_sub_level_1"],
    figsize=(6, 5),
    show_pct=True,
)

plt.show()


# Fig4e

In [ ]:
T_training = sc.read_h5ad( 'T.h5ad')


In [ ]:
T_training.layers["counts_lognorm"] = T_training.raw[:, T_training.var_names].X.copy()
sc.pp.normalize_total(T_training, target_sum=1e4, layer="counts_lognorm")
sc.pp.log1p(T_training, layer="counts_lognorm")

T_training.X = T_training.layers["counts_lognorm"]


In [ ]:
hvg_mask = T_training.var["highly_variable_features"]
T_training_hvg = T_training[:, hvg_mask].copy()
T_training_hvg.layers.clear()
T_training_hvg.raw = None

training_ref   = T_training_hvg[T_training_hvg.obs['T_sub_level_1'] != 'Pro_T'].copy()
training_ref.obs['celltype_for_training'] = training_ref.obs['T_sub_level_2'].copy()
del T_training

def to_float32_sparse(adata):
    X = adata.X
    if sp.issparse(X):
        adata.X = X.astype(np.float32).tocsr()
    else:
        adata.X = sp.csr_matrix(X.astype(np.float32))

to_float32_sparse(training_ref)

ct_model = train(
    X=training_ref,
    labels='celltype_for_training',
    feature_selection=False,
    with_mean=False,
    check_expression=False,
    max_iter=2000,
    n_jobs=4
)
ct_model.write('models/celltypist_without_ProT.pkl')


In [ ]:
ct_model = models.Model.load('models/celltypist_without_ProT.pkl')

Pro_T_query =sc.read_h5ad( 'Pro_T.h5ad')
Pro_T_query.layers["counts_lognorm"] = Pro_T_query.raw[:, Pro_T_query.var_names].X.copy()
sc.pp.normalize_total(Pro_T_query, target_sum=1e4, layer="counts_lognorm")
sc.pp.log1p(Pro_T_query , layer="counts_lognorm")

Pro_T_query.X = Pro_T_query .layers["counts_lognorm"]


In [ ]:
res_mv = celltypist.annotate(
    Pro_T_query,
    model=ct_model,
    majority_voting=True
)
adata_mv = res_mv.to_adata()
pred_mv  = adata_mv.obs['majority_voting'].astype(str)
conf_mv  = adata_mv.obs['conf_score'].astype(float)


In [ ]:
Pro_T = sc.read_h5ad( 'Pro_T.h5ad')
Pro_T.obs['label_transfer_score'] = np.nan
Pro_T.obs.loc[adata_mv.obs_names, 'label_transfer']       = pred_mv.values
Pro_T.obs.loc[adata_mv.obs_names, 'label_transfer_score'] = conf_mv.values


In [ ]:
prot_color_map = {
    "CD8_TN":    "#689f38",
    "CD8_TEM":   "#ffc107",
    "CD8_TRM":   "#8d6e63",
    "CD8_TNK":  "#e53935",
    "CD4_TH":    "#fb8c00",
    "CD4_TFH":   "#ba68c8",
    "CD4_Treg":  "#4527a0",
    "IM_T":  "#98df8a",
}


In [ ]:
ax = sc.pl.umap(
    Pro_T,
    color="label_transfer",
    palette=prot_color_map,
    use_raw=False,
    frameon=False,
    size=25,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('figures/Fig4e_ProT_label_transfer_umap.pdf', dpi=600, bbox_inches="tight")


# Fig4f

In [ ]:
marker_genes_dict = {
    "CD8/CD4":["CD8A","CD4"],
    "Naive": ["TCF7","LEF1"],
    "Activation": ["CD69","HLA-DRB1"],
    "Resident": ["ITGAE","ITGA1"],
    "Cytolytic": ["GZMK","GZMB"],
    "Terminal":["ZEB2","TBX21"],
    "NK-like":["KIR2DL4","KLRD1"],
    "Helper":["CD40LG","ICOS"],
    "Follicular Helper":["CXCL13","PDCD1"],
    "Regulatory":["FOXP3","IL2RA"],
    "Immature":["RAG1","DNTT"]
}


In [ ]:
available_genes = set(Pro_T.var_names)

filtered_marker_genes_dict = {
    category: [gene for gene in genes if gene in available_genes]
    for category, genes in marker_genes_dict.items()
}

filtered_marker_genes_dict = {
    k: v for k, v in filtered_marker_genes_dict.items() if v
}

if not filtered_marker_genes_dict:
    raise ValueError("没有找到可以用于绘图的基因，请检查 marker_genes_dict 和数据集中的基因名是否匹配。")

ax=sc.pl.dotplot(
    Pro_T,
    filtered_marker_genes_dict,
    'label_transfer',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.savefig('figures/Fig4f_ProT_markers.pdf', bbox_inches="tight")


# Fig4g

In [ ]:
def _lighten_color(color, amount=0.25):
    c = mc.to_rgb(color)
    return tuple((1 - amount) * np.array(c) + amount * np.array([1, 1, 1]))

def plot_sankey_matplotlib(
    adata,
    source_col="Period_merged",
    target_col="label_transfer",
    min_count=1,
    normalize=None,
    source_order=None,
    target_order=None,

    source_color_map=None,
    target_color_map=None,

    color_by="source",

    target_stack_by_source_order=True,

    figsize=(11, 6.5),
    node_width=0.06,
    node_pad=0.02,
    flow_alpha=0.75,
    node_alpha=0.95,
    fontsize=12,
    label_side_offset=0.01,
    curvature=0.5,
    save=None,
    dpi=300,
):

    if source_col not in adata.obs.columns:
        raise KeyError(f"{source_col} not in adata.obs")
    if target_col not in adata.obs.columns:
        raise KeyError(f"{target_col} not in adata.obs")
    if color_by not in {"source", "target"}:
        raise ValueError("color_by must be 'source' or 'target'")

    df = adata.obs[[source_col, target_col]].copy()
    df = df.dropna(subset=[source_col, target_col])
    df[source_col] = df[source_col].astype(str)
    df[target_col] = df[target_col].astype(str)

    ct = df.groupby([source_col, target_col]).size().reset_index(name="value")

    if min_count is not None and min_count > 1:
        ct = ct.loc[ct["value"] >= int(min_count)].copy()

    if ct.shape[0] == 0:
        raise ValueError("No flows left after filtering. Try smaller min_count.")

    if normalize is not None:
        normalize = normalize.lower()
        if normalize == "source":
            ct["value"] = ct["value"] / ct.groupby(source_col)["value"].transform("sum")
        elif normalize == "target":
            ct["value"] = ct["value"] / ct.groupby(target_col)["value"].transform("sum")
        elif normalize == "all":
            ct["value"] = ct["value"] / ct["value"].sum()
        else:
            raise ValueError("normalize must be None/'source'/'target'/'all'")

    def _order(col, provided):
        if provided is not None:
            return list(map(str, provided))
        s = adata.obs[col]
        if pd.api.types.is_categorical_dtype(s):
            return [str(x) for x in s.cat.categories]
        return list(ct.groupby(col)["value"].sum().sort_values(ascending=False).index)

    source_order = _order(source_col, source_order)
    target_order = _order(target_col, target_order)

    ct = ct[ct[source_col].isin(source_order) & ct[target_col].isin(target_order)].copy()

    s_tot = ct.groupby(source_col)["value"].sum().reindex(source_order).fillna(0)
    t_tot = ct.groupby(target_col)["value"].sum().reindex(target_order).fillna(0)

    source_order = [s for s in source_order if s_tot.get(s, 0) > 0]
    target_order = [t for t in target_order if t_tot.get(t, 0) > 0]
    s_tot = s_tot.reindex(source_order)
    t_tot = t_tot.reindex(target_order)

    total_pad_s = node_pad * (len(source_order) - 1) if len(source_order) > 1 else 0
    total_pad_t = node_pad * (len(target_order) - 1) if len(target_order) > 1 else 0

    s_scale = (1.0 - total_pad_s) / s_tot.sum()
    t_scale = (1.0 - total_pad_t) / t_tot.sum()

    s_y = {}
    y = 1.0
    for s in source_order:
        h = s_tot[s] * s_scale
        y1 = y
        y0 = y - h
        s_y[s] = (y0, y1)
        y = y0 - node_pad

    t_y = {}
    y = 1.0
    for t in target_order:
        h = t_tot[t] * t_scale
        y1 = y
        y0 = y - h
        t_y[t] = (y0, y1)
        y = y0 - node_pad

    base = plt.get_cmap("tab20").colors

    def _build_color_map(levels, user_map=None, lighten_amount=0.15):
        user_map = {} if user_map is None else {str(k): v for k, v in user_map.items()}
        out = {}

        for lv in levels:
            if lv in user_map:
                out[lv] = user_map[lv]

        missing = [lv for lv in levels if lv not in out]
        if len(missing) > 0:
            used = set(str(v).lower() for v in out.values())
            auto_pool = [
                _lighten_color(base[i % len(base)], lighten_amount)
                for i in range(max(len(levels), len(base)))
            ]
            auto_pool = [c for c in auto_pool if str(c).lower() not in used]

            if len(auto_pool) == 0:
                auto_pool = [
                    _lighten_color(base[i % len(base)], lighten_amount)
                    for i in range(max(len(levels), len(base)))
                ]

            for i, lv in enumerate(missing):
                out[lv] = auto_pool[i % len(auto_pool)]

        return out

    s_color = _build_color_map(source_order, source_color_map, lighten_amount=0.15)
    t_color = _build_color_map(target_order, target_color_map, lighten_amount=0.25)

    def _link_color(row):
        return s_color[row[source_col]] if color_by == "source" else t_color[row[target_col]]

    left_segments = {}
    t_rank = {t: i for i, t in enumerate(target_order)}

    for s in source_order:
        tmp = ct.loc[ct[source_col] == s].copy()
        tmp["_t_order"] = tmp[target_col].map(t_rank)
        tmp = tmp.sort_values("_t_order", ascending=True)

        cur = s_y[s][1]
        for _, row in tmp.iterrows():
            t = row[target_col]
            v = float(row["value"])
            h = v * s_scale
            y1 = cur
            y0 = cur - h
            left_segments[(s, t)] = (y0, y1)
            cur = y0

    right_segments = {}
    s_rank = {s: i for i, s in enumerate(source_order)}

    for t in target_order:
        tmp = ct.loc[ct[target_col] == t].copy()
        tmp["_s_order"] = tmp[source_col].map(s_rank)

        if target_stack_by_source_order:
            tmp = tmp.sort_values("_s_order", ascending=True)

            cur = t_y[t][1]
            for _, row in tmp.iterrows():
                s = row[source_col]
                v = float(row["value"])
                h = v * t_scale
                y1 = cur
                y0 = cur - h
                right_segments[(s, t)] = (y0, y1)
                cur = y0
        else:
            tmp = tmp.sort_values("_s_order", ascending=True)

            cur = t_y[t][0]
            for _, row in tmp.iterrows():
                s = row[source_col]
                v = float(row["value"])
                h = v * t_scale
                y0 = cur
                y1 = cur + h
                right_segments[(s, t)] = (y0, y1)
                cur = y1

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    xL0 = 0.08
    xL1 = xL0 + node_width
    xR1 = 0.92
    xR0 = xR1 - node_width

    def _bezier_band(x0, x1, y0a, y0b, y1a, y1b, curv=0.5):
        dx = x1 - x0
        cx0 = x0 + dx * curv
        cx1 = x1 - dx * curv

        verts_top = [
            (x0, y0b),
            (cx0, y0b),
            (cx1, y1b),
            (x1, y1b),
        ]
        verts_bot = [
            (x1, y1a),
            (cx1, y1a),
            (cx0, y0a),
            (x0, y0a),
        ]

        verts = [verts_top[0]] + verts_top[1:] + verts_bot + [(x0, y0b)]
        codes = [
            Path.MOVETO,
            Path.CURVE4, Path.CURVE4, Path.CURVE4,
            Path.LINETO,
            Path.CURVE4, Path.CURVE4, Path.CURVE4,
            Path.CLOSEPOLY
        ]
        verts[-1] = (x0, y0b)
        return Path(verts, codes)

    draw_ct = ct.copy()
    draw_ct["_s_order"] = draw_ct[source_col].map({s: i for i, s in enumerate(source_order)})
    draw_ct["_t_order"] = draw_ct[target_col].map({t: i for i, t in enumerate(target_order)})
    draw_ct = draw_ct.sort_values(["_s_order", "_t_order"], ascending=[True, True])

    for _, row in draw_ct.iterrows():
        s = row[source_col]
        t = row[target_col]

        y0a, y0b = left_segments[(s, t)]
        y1a, y1b = right_segments[(s, t)]

        path = _bezier_band(xL1, xR0, y0a, y0b, y1a, y1b, curv=curvature)
        patch = PathPatch(
            path,
            facecolor=_link_color(row),
            edgecolor="none",
            alpha=flow_alpha
        )
        ax.add_patch(patch)

    for s in source_order:
        y0, y1 = s_y[s]
        rect = Rectangle(
            (xL0, y0), node_width, (y1 - y0),
            facecolor=s_color[s],
            edgecolor="none",
            lw=0.0,
            alpha=node_alpha
        )
        ax.add_patch(rect)
        ax.text(
            xL0 - label_side_offset, (y0 + y1) / 2,
            s, ha="right", va="center", fontsize=fontsize
        )

    for t in target_order:
        y0, y1 = t_y[t]
        rect = Rectangle(
            (xR0, y0), node_width, (y1 - y0),
            facecolor=t_color[t],
            edgecolor="none",
            lw=0.0,
            alpha=node_alpha
        )
        ax.add_patch(rect)
        ax.text(
            xR1 + label_side_offset, (y0 + y1) / 2,
            t, ha="left", va="center", fontsize=fontsize
        )

    if save is not None:
        fig.savefig(save, dpi=dpi, bbox_inches="tight")

    plt.show()
    return fig, ax, ct


In [ ]:
Pro_T_tumor= Pro_T[Pro_T.obs["Sample_Type"] == "T"].copy()


In [ ]:
fig, ax, ct = plot_sankey_matplotlib(
    Pro_T_tumor,
    source_col="Disease",
    target_col="label_transfer",
    source_color_map=meta_color_map,
    target_color_map=prot_color_map,
    flow_alpha=0.40,
    node_alpha=0.60,
    min_count=0,
    normalize=None,
    color_by="source",
    figsize=(12, 8),
    node_pad=0.0,
    target_stack_by_source_order=True,
    save='figures/Fig4g_ProT_sankey.pdf'
)
